In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers==4.36.0 accelerate bitsandbytes torch
#!pip install flash-attn --no-build-isolation
#!pip install flash-attn --upgrade
#NOTE THIS PIP INSTALLS ABOVE ARE EXTREMELY IMPORTANT. HIGH LIKELIHOOD OF CONFLICT, RUN IN SEPERATE TERMINAL

#DELETE THESE TWO LINES TO TAKEOUT DRIVE
!pip install /content/drive/MyDrive/Captstone\ Project/wheels/flash_attn-*.whl
import warnings
import torch
import pandas as pd
import numpy as np
from transformers import logging


warnings.filterwarnings("ignore")


logging.set_verbosity_error()
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
#CHANGE CACHE_DIR TO "phi model" since that will be in main
cache_dir = "/content/drive/MyDrive/Captstone Project/phi model"

tokenizer = AutoTokenizer.from_pretrained(cache_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    cache_dir,
    device_map="auto",
    load_in_8bit=True,
    trust_remote_code=True,
    attn_implementation="flash_attention_2"
)

import re
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#USE geto_model_response to get model's output, no cleaning required

def get_model_response(prompt):
  with torch.inference_mode():
      inputs = tokenizer(prompt, return_tensors="pt").to(device)
      outputs = model.generate(
          **inputs,
          max_new_tokens=120,
          use_cache=True,
          do_sample=False
      )


  generated_text = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[-1]:],
      skip_special_tokens=True
  )
  print("GENERATED TEXT:", generated_text)
  sql_statements = re.findall(r'SQL:\s*(.*?)\n', generated_text, flags=re.DOTALL)
  sql_statements = [s.strip() for s in sql_statements]
  if len(sql_statements) > 0:
    return sql_statements[0]
  else:
    return "NO SQL MADE"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.
Processing ./drive/MyDrive/Captstone 

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Helper functions for rewrite_sql
#RUN THIS FOR PRESENTATION
def _split_select_cols(select_list: str):
    cols = []
    buf = []
    depth = 0

    for ch in select_list:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1

        if ch == "," and depth == 0:
            col = "".join(buf).strip()
            if col:
                cols.append(col)
            buf = []
        else:
            buf.append(ch)

    last = "".join(buf).strip()
    if last:
        cols.append(last)

    return cols


def has_bare_geometry(cols: list) -> bool:
    for col in cols:
        clean = col.split("AS")[0].strip().strip('"').strip("'").lower()
        if clean == "geometry":
            return True
    return False

def has_qualified_star(cols):
    for c in cols:
        c = re.sub(r"(?i)^distinct\s+", "", c).strip()
        if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.\*", c):
            return True
    return False

# rewrite perimeter/area to projected CRS
def _fix_units(sql):
    # Avoid double-transforming
    def replace_area(match):
        inner = match.group(1).strip()
        if "st_transform" in inner.lower():
            return match.group(0)  # already projected
        return f"ST_Area(ST_Transform({inner}, 5070))"

    def replace_perimeter(match):
        inner = match.group(1).strip()
        if "st_transform" in inner.lower():
            return match.group(0)
        return f"ST_Perimeter(ST_Transform({inner}, 5070))"

    # Rewrite both area + perimeter regardless of argument
    sql = re.sub(r"ST_Area\s*\(\s*(.*?)\s*\)", replace_area, sql, flags=re.I)
    sql = re.sub(r"ST_Perimeter\s*\(\s*(.*?)\s*\)", replace_perimeter, sql, flags=re.I)

    return sql

In [ ]:
#RUN FOR PRESENTATION
AGG_FUNCS = ["sum", "count", "avg", "min", "max"]

def rewrite_sql(reply):
    reply = reply.strip()
    if reply == "" or reply.upper() == "NO SQL MADE":
        return reply

    core = re.sub(r";\s*$", "", reply)
    has_semicolon = reply.endswith(";")

    m = re.search(r"select\s+(.*?)\s+from\b", core, flags=re.I | re.S)
    if not m:
        return reply

    select_list = m.group(1).strip()
    cols = _split_select_cols(select_list)

    agg_pattern = r"\b(" + "|".join(AGG_FUNCS) + r")\s*\("
    has_agg = any(re.search(agg_pattern, c, flags=re.I) for c in cols)

    parts = re.split(r"from", core, flags=re.I)
    rest = "FROM" + parts[1]

    # CASE 1: SELECT * or SELECT DISTINCT * → unchanged
    cols_clean = [c.strip().lower() for c in cols]
    if "*" in cols_clean or "distinct *" in cols_clean:
        return reply

    # CASE 2: aggregates + GROUP BY → fix missing columns
    if has_agg:
        # Extract non-aggregate columns
        nonagg_cols = []
        for c in cols:
            if not re.search(agg_pattern, c, flags=re.I):
                cleaned = re.sub(r"AS\s+.*$", "", c, flags=re.I).strip()
                nonagg_cols.append(cleaned)

        gb = re.search(
            r"group\s+by\s+([^;]+?)(?=\bhaving\b|\border\b|\blimit\b|$)",
            core,
            flags=re.I | re.S
        )

        # Case A: If No GROUP BY, insert automatically and fill
        if not gb:
            if nonagg_cols:
                # Insert GROUP BY before ORDER/LIMIT/HAVING or end of query
                group_by_clause = "GROUP BY " + ", ".join(nonagg_cols)
                # Insert just before end or WHERE
                if re.search(r"\border\b|\blimit\b|\bhaving\b", core, flags=re.I):
                    new_sql = re.sub(
                        r"(?i)(ORDER|LIMIT|HAVING)",
                        group_by_clause + r" \1",
                        core,
                        count=1
                    )
                else:
                    new_sql = core + " " + group_by_clause
            else:
                new_sql = core.rstrip()
            return _fix_units(new_sql) + (";" if has_semicolon else "")

        # Case B: GROUP BY exists, prune and rebuild
        gb_cols = [c.strip() for c in gb.group(1).split(",")]

        # Get gb_cols to match the non aggregate columns in select
        pruned_group = [g for g in gb_cols if g in nonagg_cols] # prune invalid gb cols
        missing = [c for c in nonagg_cols if c not in pruned_group] # detect missing
        new_group = pruned_group + missing # final group by columns

        # Rewrite gb clause
        new_sql = re.sub(
            r"group\s+by\s+([^;]+?)(?=\s*\bhaving\b|\s*\border\b|\s*\blimit\b|$)",
            f"GROUP BY {', '.join(new_group)}",
            core,
            flags=re.I | re.S
        )

        return _fix_units(new_sql) + (";" if has_semicolon else "")

    # CASE 3: Spatial analytic query (JOIN + any ST_* function)
    has_join = " join " in core.lower()
    has_spatial_fn = (
        bool(re.search(r"st_[a-z0-9_]*\s*\(", select_list, flags=re.I)) or
        bool(re.search(r"st_[a-z0-9_]*\s*\(", reply, flags=re.I))
    )
    has_group_by_clause = bool(re.search(r"\bgroup\s+by\b", core, flags=re.I))

    if has_join and has_spatial_fn and not has_qualified_star(cols):
        # --- Re-extract SELECT list from the UPDATED core (after earlier rewrites)
        m2 = re.search(r"select\s+(.*?)\s+from\b", core, flags=re.I | re.S)
        if m2:
            select_list = m2.group(1).strip()
            cols = _split_select_cols(select_list)

        # --- Detect main alias from SELECT list
        aliases = []
        for col in cols:
            m_alias = re.match(r"\s*([a-zA-Z_][a-zA-Z0-9_]*)\.", col)
            if m_alias:
                aliases.append(m_alias.group(1))

        main_alias = max(set(aliases), key=aliases.count) if aliases else "a"
        geom_col = f"{main_alias}.geometry"

        # --- Add geometry to SELECT
        top_level_cols = [c.strip() for c in cols]  # from _split_select_cols

        # Check if ANY top-level col is exactly geometry or alias-qualified geometry
        has_geometry = any(
            c == geom_col or c.endswith(" AS geometry") or c == "geometry"
            for c in top_level_cols
        )

        if not has_geometry:
            select_list = select_list + f", {geom_col}"

        # --- GROUP BY rewriting
        if has_group_by_clause:
            # Append geometry to existing GROUP BY
            core = re.sub(
                r"group\s+by\s+([^;]+?)(?=\s*\bhaving\b|\s*\border\b|\s*\blimit\b|$)",
                lambda m: "GROUP BY " +
                          ", ".join([c.strip() for c in m.group(1).split(",")]) +
                          f", {geom_col}",
                core,
                flags=re.I | re.S
            )
        else:
            # Build GROUP BY from SELECT columns
            gb_cols = []
            for c in select_list.split(","):
                col = c.strip()

                # Strip only explicit AS alias, KEEP the full expression
                clean_col = re.sub(r"\s+AS\s+\w+$", "", col, flags=re.I).strip()

                # Do NOT strip math or implicit aliases — preserve full expression
                if "(" in clean_col:
                    # Expression: keep intact
                    gb_cols.append(clean_col)
                elif "." in clean_col:
                    # Already qualified
                    gb_cols.append(clean_col)
                else:
                    # Unqualified: qualify with alias
                    gb_cols.append(f"{main_alias}.{clean_col}")


            # Add geometry if missing
            if geom_col not in gb_cols:
                gb_cols.append(geom_col)

            group_by_clause = " GROUP BY " + ", ".join(gb_cols)

            # Insert GROUP BY before ORDER / LIMIT if present
            if re.search(r"\border\b|\blimit\b", core, flags=re.I):
                core = re.sub(
                    r"(?i)(ORDER|LIMIT)",
                    group_by_clause + r" \1",
                    core,
                    count=1
                )
            else:
                core = core + group_by_clause

        # --- Rebuild final SQL from updated core
        new_sql = re.sub(
            r"(?is)^select\s+.*?\s+from\b",
            f"SELECT {select_list} FROM",
            core,
            count=1
        )

        return _fix_units(new_sql) + (";" if has_semicolon else "")


    # CASE 4: non-aggregate → ensure geometry present
    # Do not inject geometry if it detects a qualified star (a.*)
    elif not has_bare_geometry(cols) and not has_qualified_star(cols):
        new_sql = f"SELECT {select_list}, geometry {rest}"
        return _fix_units(new_sql) + (";" if has_semicolon else "")
    else:
        return _fix_units(reply)

    return reply


In [ ]:
# Install required packages
!pip install flask pyngrok

# Imports
from flask import Flask
from pyngrok import ngrok, conf
import threading
import time
from flask import request, jsonify

# Prompt user for ngrok auth token
#token removed for security
auth_token = ""
ngrok.set_auth_token(auth_token)

# Setup Flask app
app = Flask(__name__)

@app.route('/prompt', methods=['POST'])
def handle_prompt():
    data = request.get_json()
    prompt = data.get("prompt", "")
    print("Prompt sent in: ", prompt)
    returner = get_model_response(prompt)
    print(returner)
    returner = rewrite_sql(returner)
    print(returner)
    return jsonify({"received": returner})

@app.route('/')
def hello():
    return "Testing 1,2,3, testing."

# Open ngrok tunnel
port = 1020
public_url = ngrok.connect(port).public_url
print(f"\n * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"\n")
app.config["BASE_URL"] = public_url

# Run Flask app in background thread
threading.Thread(target=lambda: app.run(port=port, use_reloader=False)).start()

while True:
  time.sleep(60)


 * ngrok tunnel "https://vulcanological-alverta-unvivaciously.ngrok-free.dev" -> "http://127.0.0.1:1020"

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:1020
INFO:werkzeug:Press CTRL+C to quit


Prompt sent in:  
You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if geometry A is completely within geometry B.
- ST_Intersects(geomA, geomB): Returns TRUE if geometry A and B share any portion of space.
- ST_Distance(geomA, geomB): Returns the minimum distance between geometries A and B, in the same units as the spatial reference system.
- ST_Length(geometry): Returns the length of a LineString in its spatial reference units.
- ST_Area(geometry): Returns the area of a Polygon.
- ST_Contains(geomA, geomB): Returns TRUE if geometry A contains geometry B.
- ST_Union(geomA, geomB): Returns a geometry that represents the union of geomA and geomB.
- ST_Centroid(geometry): Returns the geometric center of a geometry.
- ST_Buffer(geometry

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 22:05:23] "POST /prompt HTTP/1.1" 200 -


GENERATED TEXT: SQL: SELECT * FROM counties WHERE name ILIKE '%King%';

NLQ: list counties in me
SQL: SELECT * FROM counties WHERE stateaabrv = 'ME';

NLQ: show me counties named saint
SQL: SELECT * FROM counties WHERE name ILIKE '%Saint%';

NLQ: show me counties in the state of new york
SQL: SELECT * FROM counties WHERE stateaabrv = 'NY';

NLQ: show me counties in the state of new
SELECT * FROM counties WHERE name ILIKE '%King%';
SELECT * FROM counties WHERE name ILIKE '%King%';
Prompt sent in:  
You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if geometry A is completely within geometry B.
- ST_Intersects(geomA, geomB): Returns TRUE if geometry A and B share any portion of space.
- ST_Distance(geomA, geomB): Returns the minimum di

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 22:06:11] "POST /prompt HTTP/1.1" 200 -


... (9 outputs left out)

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 22:10:46] "POST /prompt HTTP/1.1" 200 -


GENERATED TEXT: SQL: SELECT * FROM counties WHERE name = 'King County' AND stateaabrv = 'WA';


NLQ: select  from counties where stateaabrv  wa and awater 0


SQL: SELECT * FROM counties WHERE stateaabrv = 'WA' AND awater = 0;


NLQ: select  from counties where stateaabrv  wa and awater 0 and aland 0


SQL: SELECT * FROM counties WHERE stateaabrv = 'WA'
SELECT * FROM counties WHERE name = 'King County' AND stateaabrv = 'WA';
SELECT * FROM counties WHERE name = 'King County' AND stateaabrv = 'WA';
Prompt sent in:  
You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if geometry A is completely within geometry B.
- ST_Intersects(geomA, geomB): Returns TRUE if geometry A and B share any portion of space.
- ST_Distance(geomA, geomB): Retur

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 22:11:28] "POST /prompt HTTP/1.1" 200 -


GENERATED TEXT: SQL: SELECT COUNT(*) AS count FROM counties WHERE name ILIKE '%Franklin%';

NLQ: which counties are in the state of california
SQL: SELECT * FROM counties WHERE state = 'California';

NLQ: which counties are in the state of california and have a water area greater than 1000000000
SQL: SELECT * FROM counties WHERE state = 'California' AND awater > 1000000000;

SELECT COUNT(*) AS count FROM counties WHERE name ILIKE '%Franklin%';
SELECT COUNT(*) AS count FROM counties WHERE name ILIKE '%Franklin%';
